In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# This notebook is meant for extracting Knowledge Graphs from MMD Files

In [4]:
import llm
import helpers
import torch
import outlines
import os
import pandas as pd
from outlines import models
from langchain_community.graphs import Neo4jGraph

# Configure these parameters to select the appropriate graph and prompting strategy to use

In [5]:
from config import NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD,NEO4J_DATABASE, DIRECTORY
graph = Neo4jGraph(NEO4J_URI,NEO4J_USERNAME,NEO4J_PASSWORD,NEO4J_DATABASE, refresh_schema=False)

/tmp/ipykernel_716205/2808298265.py:2: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(NEO4J_URI,NEO4J_USERNAME,NEO4J_PASSWORD,NEO4J_DATABASE, refresh_schema=False)


In [6]:
# clear all nodes in the graph if not empty
graph.query("MATCH (n) DETACH DELETE n")

[]

In [7]:
# Drop indexes
graph.query("DROP INDEX vector IF EXISTS")

# Drop all constraints
constraints = graph.query("SHOW CONSTRAINTS")
for c in constraints:
    graph.query(f"DROP CONSTRAINT {c['name']} IF EXISTS")

# Drop all indexes  
indexes = graph.query("SHOW INDEXES")
for idx in indexes:
    graph.query(f"DROP INDEX {idx['name']} IF EXISTS")

In [8]:
graph.query("SHOW INDEXES")

[]

## Markdown parsing and chunking

In [9]:
import markdown_parser
from pathlib import Path

chunks = []
files = Path(DIRECTORY).glob('**/*.md')
for file in files:
    print(file)
    if os.path.isfile(file):
        # simple markdown parser that removes citations, urls, references, acknowledgements, and basically everything after the conclusion
        content = markdown_parser.process_markdown_paper(str(file))
        # semantic chunk is chunking w.r.t sentences, and has overlap param as well
        chunks.extend(markdown_parser.semantic_chunk(content) )

../clbp_causal_md/hu2022/hu2022.md
../clbp_causal_md/amiri/amiri.md
../clbp_causal_md/lancet2020/lancet2020.md
../clbp_causal_md/faravelli2013/faravelli2013.md
../clbp_causal_md/farmer2019/farmer2019.md
../clbp_causal_md/marshall2018/marshall2018.md
../clbp_causal_md/wettstein2019/wettstein2019.md
../clbp_causal_md/nida/nida.md
../clbp_causal_md/kanel2022/kanel2022.md
../clbp_causal_md/williams2022/williams2022.md
../clbp_causal_md/almeida/almeida.md
../clbp_causal_md/markfelder2020/markfelder2020.md
../clbp_causal_md/zhao2022/zhao2022.md
../clbp_causal_md/zhu2020/zhu2022.md
../clbp_causal_md/fluharty2017/fluharty2017.md
../clbp_causal_md/kohler2018/kohler2018.md
../clbp_causal_md/salive2013/salive2013.md
../clbp_causal_md/lenze2000/lenze2000.md
../clbp_causal_md/ohagan2023/ohagan2023.md
../clbp_causal_md/zhouj2021/zhouj2021.md
../clbp_causal_md/assari/assari.md
../clbp_causal_md/guan2022/guan2022.md
../clbp_causal_md/edwards2004/edwards2004.md
../clbp_causal_md/zhou2022/zhou2022.md
..

In [10]:
len(chunks)

5758

## Triplet Extraction

In [11]:
from typing import List
from langchain_community.graphs.graph_document import GraphDocument
from langchain_core.documents import Document
from retry import retry
from tqdm import tqdm
from models.KnowledgeGraphSchema import KnowledgeGraph
from prompts import graph_extraction_prompt

llm_transformer = llm.LLMGraphTransformer(
    schema=KnowledgeGraph,
    prompt=graph_extraction_prompt
)

In [12]:
# dispatches every chunk's extraction call concurrently via the backend's .map()
# instead of looping one chunk at a time - a big win against a hosted API, a no-op
# against the local vLLM server (see VLLMClient.map)
docs = [Document(page_content=chunk) for chunk in chunks[8:108]]
graph_documents = llm_transformer.convert_to_graph_documents_concurrent(docs)

APIClient.map:   0%|          | 0/100 [00:00<?, ?it/s]

In [14]:
len(graph_documents)


100

In [15]:
# this automatically combines nodes / relationships that have the same id
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

In [16]:
data = pd.DataFrame(graph.query("match (n:__Entity__)-[r]->(m:__Entity__) return n.id, n.description, r.strength, type(r), r.description, m.id, m.description order by type(r), n.id asc"))
# data.to_csv("tmp/quick_analysis.csv")
data

,n.id,n.description,r.strength,type(r),r.description,m.id,m.description
0,Interventions,"A planned, systematic action intended to modif...",1,ADDRESSES,Interventions are planned to address harmful a...,Harmful_Alcohol_Consumption,Alcohol consumption that leads to negative hea...
1,Adjustment For Confounders In Risk Estimation,Adjustments made to the relative risk estimate...,1,ADJUSTMENT,Adjustments were made to the relative risk est...,Alcohol,"Alcohol consumption, measured in number of dri..."
2,Age,The age of the patient,1,ADJUSTMENT,Age was used as a covariate to adjust for diff...,Alcohol,"Alcohol consumption, measured in number of dri..."
3,Exposure Measurement Timing,The timing of the measurement of alcohol consu...,1,ADJUSTMENT,Exposure measurement timing was used as a cova...,Alcohol,"Alcohol consumption, measured in number of dri..."
4,Geographical Representativeness,The degree to which the geographical location ...,1,ADJUSTMENT,Geographical representativeness was used as a ...,Alcohol,"Alcohol consumption, measured in number of dri..."
...,...,...,...,...,...,...,...
1467,S. Behnezhad,Researcher in the Department of Psychology at ...,1,WORKS_AT,S. Behnezhad is a researcher in this department,"Department Of Psychology, Kharazmi University,...",The department where S. Behnezhad works
1468,2014,Year when the Mendelian Randomization: genetic...,1,YEAR_OF_PUBLICATION,The year the Mendelian Randomization: genetic ...,16,None
1469,2016,Year when Mendelian randomization as an approa...,1,YEAR_OF_PUBLICATION,The year Mendelian randomization as an approac...,17,None
1470,2017,2017 is the year the review on 10 Years of GWA...,1,YEAR_OF_PUBLICATION,The year the Mendelian Randomization article w...,17,None


# Entity Resolution

In [17]:
import helpers
from langchain_community.vectorstores import Neo4jVector
from langchain_community.embeddings import OllamaEmbeddings, HuggingFaceEmbeddings
from graphdatascience import GraphDataScience

import outlines
from vllm.sampling_params import SamplingParams
from prompts import prompt_er
from pydantic import BaseModel, create_model, Field
from typing import List, Optional
from retry import retry

In [18]:
pubmedbert_embeddings = HuggingFaceEmbeddings(
    model_name="pritamdeka/S-PubMedBert-MS-MARCO",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

/tmp/ipykernel_716205/4189449969.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  pubmedbert_embeddings = HuggingFaceEmbeddings(


In [19]:
graph.query("DROP INDEX vector IF EXISTS;")

[]

In [20]:
vector = Neo4jVector.from_existing_graph(
    pubmedbert_embeddings,
    node_label='__Entity__',
    text_node_properties=['id', 'description'],
    embedding_node_property='embedding',
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

In [21]:
# project graph
gds = GraphDataScience(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD), database=NEO4J_DATABASE

)

# Drop GDS projections if they exist
try:
    gds.graph.drop("entities")
except:
    pass
try:
    gds.graph.drop("communities")
except:
    pass

In [22]:
G, result = gds.graph.project(
    "entities",                   #  Graph name
    "__Entity__",                 #  Node projection
    "*",                          #  Relationship projection
    nodeProperties=["embedding"]  #  Configuration parameters
)

In [23]:
# experiment with a higher sim threshold
similarity_threshold = 0.98

gds.knn.mutate(
  G,
  nodeProperties=['embedding'],
  mutateRelationshipType= 'SIMILAR',
  mutateProperty= 'score',
  similarityCutoff=similarity_threshold,
  randomSeed=42,
  concurrency=1,
  sampleRate=1.0,
  deltaThreshold=0.0
)

 Knn:   0%|          | 0/100 [00:00<?, ?%/s]

ranIterations                                                             6
nodePairsConsidered                                                  447349
didConverge                                                            True
preProcessingMillis                                                       0
computeMillis                                                           743
mutateMillis                                                             21
postProcessingMillis                                                      0
nodesCompared                                                           756
relationshipsWritten                                                   1696
similarityDistribution    {'min': 0.9800071716308594, 'p5': 0.9805641174...
configuration             {'randomSeed': 42, 'mutateProperty': 'score', ...
Name: 0, dtype: object

In [24]:
gds.wcc.write(
    G,
    writeProperty="wcc",
    relationshipTypes=["SIMILAR"]
)

writeMillis                                                            108
nodePropertiesWritten                                                  756
componentCount                                                         404
componentDistribution    {'min': 1, 'p5': 1, 'max': 61, 'p999': 61, 'p9...
postProcessingMillis                                                     7
preProcessingMillis                                                      0
computeMillis                                                            4
configuration            {'writeProperty': 'wcc', 'jobId': '8f5cad8c-f1...
Name: 0, dtype: object

In [25]:
edit_distance_query = helpers.load_query("edit_distance.cypher")
print(edit_distance_query)

MATCH (e:`__Entity__`)
WHERE size(e.id) > $min_length
WITH e.wcc AS community, collect(e) AS nodes, count(*) AS count
WHERE count > 1
UNWIND nodes AS node
// Add text distance
WITH distinct
[n IN nodes WHERE apoc.text.distance(toLower(node.id), toLower(n.id)) < $distance | n.id] AS intermediate_results
WHERE size(intermediate_results) > 1
WITH collect(intermediate_results) AS results
// combine groups together if they share elements
UNWIND range(0, size(results)-1, 1) as index
WITH results, index, results[index] as result
WITH apoc.coll.sort(reduce(acc = result, index2 IN range(0, size(results)-1, 1) |
    CASE WHEN index <> index2 AND
	size(apoc.coll.intersection(acc, results[index2])) > 0
	THEN apoc.coll.union(acc, results[index2])
	ELSE acc
    END
)) as combinedResult
WITH distinct(combinedResult) as combinedResult
// extra filtering
WITH collect(combinedResult) as allCombinedResults
UNWIND range(0, size(allCombinedResults)-1, 1) as combinedResultIndex
WITH allCombinedResults[combi

In [26]:
word_edit_distance = 2
potential_duplicate_candidates = graph.query(edit_distance_query, params={'distance': word_edit_distance, 'min_length': 5})

In [27]:
from prompts import prompt_er
from llm_client import get_client

class DuplicateEntities(BaseModel):
    entities: List[str] = Field(
        description="Entities that represent the same object or real-world entity and should be merged"
    )


class Disambiguate(BaseModel):
    merge_entities: Optional[List[DuplicateEntities]] = Field(
        description="Lists of entities that represent the same object or real-world entity and should be merged"
    )

extraction_llm = get_client(schema=Disambiguate)

@retry(tries=1, delay=2)
def entity_resolution(entities: List[str]) -> Optional[List[str]]:
    res = extraction_llm(prompt_er(sorted(entities)), sampling_params={"n":1, "temperature":0, "top_k":1})
    return [
        el.entities
        for el in res.merge_entities
    ]

In [28]:
from tqdm import tqdm 

merged_entities = []

for el in tqdm(potential_duplicate_candidates, total=len(potential_duplicate_candidates), desc="Resolving entities"):
    merged_entities.extend(entity_resolution(el["combinedResult"]))

Resolving entities:   0%|          | 0/16 [00:00<?, ?it/s]

Resolving entities: 100%|██████████| 16/16 [00:07<00:00,  2.22it/s]


In [30]:
graph.query("""
UNWIND $data AS candidates
CALL {
  WITH candidates
  MATCH (e:__Entity__) WHERE e.id IN candidates
  RETURN collect(e) AS nodes
}
CALL apoc.refactor.mergeNodes(nodes, {properties: {
    `.*`: 'discard'
}})
YIELD node
RETURN count(*)
""", params={"data": merged_entities})

[{'count(*)': 17}]

In [31]:
gds.graph.drop("entities")

graphName                                                         entities
database                                                             neo4j
databaseLocation                                                     local
memoryUsage                                                               
sizeInBytes                                                             -1
nodeCount                                                              756
relationshipCount                                                     3168
configuration            {'relationshipProjection': {'__ALL__': {'aggre...
density                                                            0.00555
creationTime                           2026-07-18T22:02:19.341112751+00:00
modificationTime                       2026-07-18T22:02:21.515261308+00:00
schema                   {'graphProperties': {}, 'nodes': {'__Entity__'...
schemaWithOrientation    {'graphProperties': {}, 'nodes': {'__Entity__'...
Name: 0, dtype: object

In [32]:
G.drop()

,graphName,database,databaseLocation,memoryUsage,sizeInBytes,nodeCount,relationshipCount,configuration,density,creationTime,modificationTime,schema,schemaWithOrientation


# Communities

In [33]:
gds.graph.drop('communities')

In [34]:
G, result = gds.graph.project(
    "communities",  #  Graph name
    "__Entity__",  #  Node projection
    {
        "_ALL_": {
            "type": "*",
            "orientation": "UNDIRECTED",
            "properties": {"weight": {"property": "*", "aggregation": "COUNT"}},
        }
    },
)

In [35]:
wcc = gds.wcc.stats(G)
print(f"Component count: {wcc['componentCount']}")
print(f"Component distribution: {wcc['componentDistribution']}")

Component count: 99
Component distribution: {'min': 1, 'p5': 1, 'max': 471, 'p999': 471, 'p99': 471, 'p1': 1, 'p10': 1, 'p90': 7, 'p50': 1, 'p25': 1, 'p75': 3, 'p95': 12, 'mean': 7.383838383838384}


In [36]:
gds.leiden.write(
    G,
    writeProperty="communities",
    includeIntermediateCommunities=True,
    relationshipWeightProperty="weight",
    randomSeed=42,
    concurrency=1,
    theta=0
)

writeMillis                                                             31
nodePropertiesWritten                                                  731
ranLevels                                                                4
didConverge                                                           True
nodeCount                                                              731
communityCount                                                         112
communityDistribution    {'min': 1, 'p5': 1, 'max': 112, 'p999': 112, '...
modularity                                                          0.5976
modularities             [0.545253241670605, 0.5910088407520085, 0.5975...
postProcessingMillis                                                     1
preProcessingMillis                                                      0
computeMillis                                                           57
configuration            {'writeProperty': 'communities', 'randomSeed':...
Name: 0, dtype: object

In [37]:
graph.query("CREATE CONSTRAINT IF NOT EXISTS FOR (c:__Community__) REQUIRE c.id IS UNIQUE;")

[]

In [38]:
# creates community nodes and adds edges from each node to the community it belongs too. 
graph.query("""
MATCH (e:`__Entity__`)
UNWIND range(0, size(e.communities) - 1 , 1) AS index
CALL {
  WITH e, index
  WITH e, index
  WHERE index = 0
  MERGE (c:`__Community__` {id: toString(index) + '-' + toString(e.communities[index])})
  ON CREATE SET c.level = index
  MERGE (e)-[:IN_COMMUNITY]->(c)
  RETURN count(*) AS count_0
}
CALL {
  WITH e, index
  WITH e, index
  WHERE index > 0
  MERGE (current:`__Community__` {id: toString(index) + '-' + toString(e.communities[index])})
  ON CREATE SET current.level = index
  MERGE (previous:`__Community__` {id: toString(index - 1) + '-' + toString(e.communities[index - 1])})
  ON CREATE SET previous.level = index - 1
  MERGE (previous)-[:IN_COMMUNITY]->(current)
  RETURN count(*) AS count_1
}
RETURN count(*)
""")

[{'count(*)': 2924}]

In [39]:
graph.query("MATCH (n:`__Community__`) return count(n)")

[{'count(n)': 526}]

In [40]:
# community structure
# finds the size of all communities with more than two entities
community_size_df = graph.query(
    """
    MATCH (c:__Community__)<-[:IN_COMMUNITY*]-(e:__Entity__)
    WITH c, count(distinct e) AS entities
    WHERE entities > 1
    RETURN split(c.id, '-')[0] AS level, entities
    """
)
community_size_df = pd.DataFrame(community_size_df)
helpers.community_analysis(community_size_df)

,Level,Number of communities,25th Percentile,50th Percentile,75th Percentile,90th Percentile,99th Percentile,Max
0,0,125,2.0,3.0,4.0,7.0,51.04,112
1,1,69,2.0,4.0,8.0,15.4,102.48,112
2,2,58,2.0,4.0,8.0,19.9,106.30,112
3,3,58,2.0,4.0,8.0,24.1,106.30,112


# Summarizing Communities

In [41]:
community_info = graph.query("""
MATCH (c:`__Community__`)<-[:IN_COMMUNITY*]-(e:__Entity__)
WITH c, collect(e) AS nodes
WHERE size(nodes) > 1
CALL apoc.path.subgraphAll(nodes[0], {
	whitelistNodes:nodes
})
YIELD relationships
RETURN c.id AS communityId, c.level as level,
       [r in relationships | {
                             start_id: startNode(r).id,
                             start_desc: startNode(r).description, 
                             rel_desc: r.description, 
                             rel_type: type(r),
                             end_id: endNode(r).id,
                             end_desc: endNode(r).description, 
                             degree: apoc.node.degree(startNode(r)) + apoc.node.degree(endNode(r))
                             }] AS triplets
""")

In [42]:
# first pass: sort the triplets, separate into two groups: leaves and nonleaves
context_window_limit = 8000

In [44]:
from build_context import split_and_sort, summarize_leaves, normalize_nonleaves, summarize_nonleaves

raw_leaves, raw_nonleaves = split_and_sort(community_info) 
leaves_with_reports, leaves_reports_map = summarize_leaves(raw_leaves, context_window_limit)

Building Leaf Contexts: 100%|██████████| 125/125 [00:00<00:00, 1384.71it/s]


APIClient.map:   0%|          | 0/125 [00:00<?, ?it/s]

In [45]:
nonleaves_normalized = normalize_nonleaves(raw_nonleaves, leaves_reports_map)
nonleaves_with_reports = summarize_nonleaves(nonleaves_normalized, context_window_limit)

Normalizing Non Leaves


Nonleaf Normalization Progress: 100%|██████████| 185/185 [00:00<00:00, 324.29it/s]


Finished Normalizing Non Leaves
Building Nonleaf Contexts


Nonleaf Context Building Progress: 100%|██████████| 185/185 [00:00<00:00, 1121.01it/s]


Summarizing Nonleaves


APIClient.map:   0%|          | 0/185 [00:00<?, ?it/s]

Finished Summarizing Non Leaves


In [46]:
from build_context import normalize_summarized_community

leaves = list(
    map(normalize_summarized_community, leaves_with_reports)
)

nonleaves = list(
    map(normalize_summarized_community, nonleaves_with_reports)
)

In [47]:
# Store summaries
graph.query("""
UNWIND $data AS row
MERGE (c:__Community__ {id:row.community})
SET c.title=row.title, c.summary = row.summary, c.impact_severity_rating=row.impact_severity_rating, c.rating_explanation=row.rating_explanation, c.detailed_findings=row.detailed_findings
""", params={"data": leaves + nonleaves})

[]

# Weighting the Graph

In [48]:
# community rank set to be the number of documents referenced by that community
graph.query("""
MATCH (c:__Community__)<-[:IN_COMMUNITY*]-(:__Entity__)<-[:MENTIONS]-(d:Document)
WITH c, count(distinct d) AS rank
SET c.community_rank = rank;
""")

[]